In [ ]:
import sys
from bisect import bisect_left

def main(lines):
    # モード1と変更後のモードxにおいて生産されたお菓子の重さにおいて、それぞれのモードにて、
    # (お菓子の重さ - モードでの平均値)^2の総和の和が最小となるような、tとxを求めればよい

    # 標準入力の受け取り
    # 重さとモードはリストのまま保存する
    row0 = lines[0].split()
    row1 = lines[1].split()
    row2 = lines[2].split()

    N = int(row0[0])
    K = int(row0[1])
    sigma = float(row0[2])
    A = [float(a) for a in row1]
    M = [float(m) for m in row2]

    # モードは2番目以上が探索範囲なので、1番目と分けておく
    M1 = M[0]
    M_vals = M[1:]

    # 最終的な解答用
    best_err = float('inf')
    best_t = -1
    best_x = -1

    # 事前に総和を計算
    prefix_sum = [0] * (N + 1)
    prefix_sum_sq = [0] * (N + 1)
    for i in range(N):
        prefix_sum[i + 1] = prefix_sum[i] + A[i]
        prefix_sum_sq[i + 1] = prefix_sum_sq[i] + A[i] * A[i]

    # モードが変わった位置tを固定して、その場合での尤もらしいモードを求める
    # モード変更は1度は必ず起きていることを考慮して、tは1以上N以下
    for t in range(1, N):
        # モード1での(お菓子の重さ - モードでの平均値)^2の総和を求める
        sum_a1 = prefix_sum[t]
        sum_a1_sq = prefix_sum_sq[t]
        sum_1 = sum_a1_sq - 2 * M1 * sum_a1 + t * M1 * M1
 
        # モード変更後の平均値に最も近い値を求める
        # 高速化のため、二分探索を実装する
        sum_ax = prefix_sum[N] - prefix_sum[t]
        sum_ax_sq = prefix_sum_sq[N] - prefix_sum_sq[t]

        ave_ax = sum_ax / (N - t)
        x = bisect_left(M_vals, ave_ax)

        # このライブラリは挿入して順序を崩さない箇所のうち、最も左の箇所を返すらしいので、
        # 求められたxとx-1のM_vals、どちらが最も近いかを確かめる
        if x == 0:
            # 最も左に挿入されるので、モードの平均値はM_vals[0]
            Mx = M_vals[x]
        elif x == len(M_vals):
            # 最も右に挿入されるので、挿入箇所がリスト外になる。モードの平均値はM_vals[x-1]
            x = x - 1
            Mx = M_vals[x]
        else:
            left = M_vals[x - 1]
            right = M_vals[x]
            Mx = left if abs(left - ave_ax) <= abs(right - ave_ax) else right
            x = (x - 1) if abs(left - ave_ax) <= abs(right - ave_ax) else x
        
        sum_x = sum_ax_sq - 2 * Mx * sum_ax + (N - t) * Mx * Mx
        best_err_t = sum_1 + sum_x

        if best_err_t < best_err:
            best_err = best_err_t
            best_x = x + 2
            best_t = t + 1
    
    print(best_t, best_x)

if __name__ == '__main__':
    lines = []
    for l in sys.stdin:
        lines.append(l.rstrip('\r\n'))
    main(lines)

In [ ]:
import sys

def main(lines):
    # 最も広いスペースを用意する。
    # 左上(0,0)の位置にスタートを置いて、そこから、右に進む。
    # 右端についたら、2マス下がり、左に進む。-1
    # 左端についたら、2マス下がり、右に進む。-2
    # 1, 2を繰り返すことでkを満たすようにする

    k = int(lines[0])
    # 最も広いダンジョンを想定
    n = 50
    m = 50

    # ダンジョンの中で障害物ではない可能性のある「経路候補」を決める
    # 経路候補は、その上位から経路になっていくため、偶数行目においては、左端で降りた場合には左から、右端で降りた場合には右から、
    # のように、厳密に接続されている必要がある
    path = []
    for i in range(0, n):
        # 偶数行 - すべて経路候補であるが、4で割ったあまりが0の時は左から、2の時は右からの順に経路候補になる
        if i % 4 == 0:
            for j in range(0, m):
                path.append((i, j))
        elif i % 4 == 2:
            for j in range(m-1, -1, -1):
                path.append((i, j))
        # 奇数行 - 4で割ったあまりが1の時は右端、3の時は左端が経路候補になる
        elif i % 4 == 1:
            path.append((i, m-1))
        else:
            path.append((i, 0))
        
    # 経路はスタートを含むため、経路候補のうちk+1個
    path = path[:k+1]

    # すべて障害物のダンジョンの中から、path[0]をS、path[-1]をG、それ以外のpathを.とする
    dungeon = [["#"] * m for _ in range(n)]
    for i, (x, y) in enumerate(path):
        if i == 0:
            dungeon[x][y] = "S"
        elif i == len(path)-1: # kとも同じ
            dungeon[x][y] = "G"
        else:
            dungeon[x][y] = "."
    
    # 出力
    print(n, m)
    for row in dungeon:
        print("".join(row))


if __name__ == '__main__':
    lines = []
    for l in sys.stdin:
        lines.append(l.rstrip('\r\n'))
    main(lines)